# CARE-BED — final window-size ablation on A13 + B12

This notebook repeats the temporal-window ablation on the **final 25-recording dataset**:

```text
Environment A: 13 historical recordings
Environment B: 12 recordings = s01, s02, s03 for each activity
TOTAL:         25 independent recordings
```

Only the window length changes:

```text
W10_HSG = 10 frames + Hampel + Savitzky-Golay
W20_HSG = 20 frames + Hampel + Savitzky-Golay
W40_HSG = 40 frames + Hampel + Savitzky-Golay
```

All configurations use the **same fixed 5-fold recording-disjoint split**.

This experiment is intended only to select / justify the temporal window on the final dataset.  
The separate 25-fold LORO notebook remains the stronger final offline generalization analysis.

## 0. Install packages

In [ ]:
!pip -q install hampel joblib

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
from math import sqrt
import csv
import json
import random
import re
import shutil
import time
import zipfile

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hampel import hampel
from scipy.signal import savgol_filter

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Upload datasets

Upload:

```text
envA_dataset.zip
carebed_new_data.zip
```

In [ ]:
WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()

OLD_ZIP = WORKDIR / "envA_dataset.zip"
NEW_ZIP = WORKDIR / "carebed_new_data.zip"

if not OLD_ZIP.exists() or not NEW_ZIP.exists():
    try:
        from google.colab import files
        print("Upload envA_dataset.zip and carebed_new_data.zip")
        files.upload()
    except ImportError:
        pass

if not OLD_ZIP.exists():
    raise FileNotFoundError("Missing envA_dataset.zip")

if not NEW_ZIP.exists():
    raise FileNotFoundError("Missing carebed_new_data.zip")

OLD_ROOT = WORKDIR / "envA_dataset"
NEW_ROOT = WORKDIR / "carebed_new_data"

for root in [OLD_ROOT, NEW_ROOT]:
    if root.exists():
        shutil.rmtree(root)

with zipfile.ZipFile(OLD_ZIP, "r") as zf:
    zf.extractall(WORKDIR)

with zipfile.ZipFile(NEW_ZIP, "r") as zf:
    zf.extractall(WORKDIR)

if not OLD_ROOT.exists():
    raise FileNotFoundError(
        "envA_dataset.zip must contain top-level envA_dataset/"
    )

if not NEW_ROOT.exists():
    raise FileNotFoundError(
        "carebed_new_data.zip must contain top-level carebed_new_data/"
    )

print("Datasets extracted.")

## 3. Configuration

In [ ]:
EXPECTED_ROLE = "AP"
NEW_MAC = "24:0A:C4:02:29:F0"

OLD_SOURCE_TOKENS = {
    "inactivity": "br_ak",
    "lying down": "lezenie",
    "sitting up": "siedzenie",
    "fidgeting": "wiercenie",
}

OLD_RECORDING_RANGES = {
    "inactivity": [
        ("A_inactivity_s01", 1, 1765),
        ("A_inactivity_s02", 1766, 2427),
        ("A_inactivity_s03", 2428, 3095),
        ("A_inactivity_s04", 3096, 3202),
    ],
    "lying down": [
        ("A_lying_down_s01", 1, 607),
        ("A_lying_down_s02", 608, 1779),
        ("A_lying_down_s03", 1780, 2947),
    ],
    "sitting up": [
        ("A_sitting_up_s01", 1, 601),
        ("A_sitting_up_s02", 602, 1739),
        ("A_sitting_up_s03", 1740, 2922),
    ],
    "fidgeting": [
        ("A_fidgeting_s01", 1, 1123),
        ("A_fidgeting_s02", 1124, 1810),
        ("A_fidgeting_s03", 1811, 2955),
    ],
}

EXPECTED_OLD_LINE_COUNTS = {
    "inactivity": 3202,
    "lying down": 2947,
    "sitting up": 2922,
    "fidgeting": 2955,
}

B_FOLDERS = {
    "inactivity": "inactivity",
    "lying down": "lying_down",
    "sitting up": "sitting_up",
    "fidgeting": "fidgeting",
}

# Final balanced Environment B subset.
VALID_B_SESSIONS = {1, 2, 3}

FIRST_CSI_VALUES = 128
RAW_FEATURES = 64

COLUMNS_TO_DROP = [2, 3, 4, 5, 32, 59, 60, 61, 62, 63]
N_FEATURES = 54

STRIDE = 5
DEV_TRAIN_FRACTION = 0.80

HAMPEL_WINDOW = 10
SAVGOL_WINDOW = 10
SAVGOL_POLYORDER = 3

LEARNING_RATE = 0.0005
BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 5

N_SPLITS = 5

CONFIGS = [
    {"config_id": "W10_HSG", "window": 10},
    {"config_id": "W20_HSG", "window": 20},
    {"config_id": "W40_HSG", "window": 40},
]

PREPROCESS_CACHE = WORKDIR / "carebed_A13_B12_HSG_processed_cache.joblib"

SUMMARY_CSV = WORKDIR / "carebed_A13_B12_window_ablation_summary.csv"
FOLD_CSV = WORKDIR / "carebed_A13_B12_window_ablation_fold_metrics.csv"
PREDICTIONS_CSV = WORKDIR / "carebed_A13_B12_window_ablation_predictions.csv"

OVERALL_PDF = WORKDIR / "carebed_A13_B12_window_ablation_overall.pdf"
ENV_PDF = WORKDIR / "carebed_A13_B12_window_ablation_environment.pdf"

RESULT_ZIP = WORKDIR / "CARE_BED_A13_B12_window_ablation_results.zip"

assert RAW_FEATURES - len(COLUMNS_TO_DROP) == N_FEATURES

## 4. CSI parsing helpers

In [ ]:
def parse_bracket_payload(line):
    try:
        content = re.findall(r"\[(.*)\]", line)[0]
        return [int(x) for x in content.split(" ") if x != ""]
    except Exception:
        return None


def amplitude_from_payload(payload):
    if payload is None or len(payload) < FIRST_CSI_VALUES:
        return None

    raw = payload[:FIRST_CSI_VALUES]

    imaginary = raw[0::2]
    real = raw[1::2]

    if len(real) != RAW_FEATURES:
        return None

    return np.asarray(
        [
            sqrt(
                imaginary[i] ** 2
                + real[i] ** 2
            )
            for i in range(RAW_FEATURES)
        ],
        dtype=np.float32,
    )


def old_line_to_amplitude(line):
    return amplitude_from_payload(
        parse_bracket_payload(line)
    )


def new_line_to_amplitude(line):
    try:
        row = next(csv.reader([line]))
    except Exception:
        return None

    if len(row) != 26:
        return None

    if row[0].strip() != "CSI_DATA":
        return None

    if row[1].strip() != EXPECTED_ROLE:
        return None

    if row[2].strip().upper() != NEW_MAC.upper():
        return None

    try:
        declared_len = int(row[24].strip())
    except ValueError:
        return None

    payload = parse_bracket_payload(line)

    if payload is None or len(payload) != declared_len:
        return None

    return amplitude_from_payload(payload)


def session_number_from_name(path):
    match = re.search(
        r"_s(\d+)\.csv$",
        path.name,
        flags=re.IGNORECASE,
    )

    return int(match.group(1)) if match else None

## 5. Load A13 + B12

In [ ]:
recordings_A = []
recordings_B = []

# Environment A
old_source_files = {}

for activity, token in OLD_SOURCE_TOKENS.items():
    matches = [
        path
        for path in OLD_ROOT.glob("*.csv")
        if token in path.name.lower()
    ]

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one Environment A CSV containing '{token}', "
            f"found {[p.name for p in matches]}"
        )

    old_source_files[activity] = matches[0]


for activity, ranges in OLD_RECORDING_RANGES.items():
    source = old_source_files[activity]

    with source.open(
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as fh:
        lines = fh.readlines()

    expected = EXPECTED_OLD_LINE_COUNTS[activity]

    if len(lines) != expected:
        raise ValueError(
            f"{source.name}: expected {expected} lines, found {len(lines)}"
        )

    for recording_id, start1, end1 in ranges:
        amplitudes = [
            old_line_to_amplitude(line)
            for line in lines[start1 - 1:end1]
        ]

        amplitudes = [
            value
            for value in amplitudes
            if value is not None
        ]

        raw = pd.DataFrame(amplitudes)

        recordings_A.append({
            "recording_id": recording_id,
            "label": activity,
            "environment": "Environment A",
            "source_file": source.name,
            "raw": raw.reset_index(drop=True),
        })


# Environment B: only s01-s03
for activity, folder_name in B_FOLDERS.items():
    folder = NEW_ROOT / folder_name

    for path in sorted(folder.glob("*.csv")):
        if path.name.endswith("_timing.csv"):
            continue

        session_number = session_number_from_name(path)

        if session_number not in VALID_B_SESSIONS:
            continue

        amplitudes = []

        with path.open(
            "r",
            encoding="utf-8",
            errors="ignore",
        ) as fh:
            for line in fh:
                value = new_line_to_amplitude(line)

                if value is not None:
                    amplitudes.append(value)

        if not amplitudes:
            raise ValueError(
                f"{path.name}: no valid CSI frames"
            )

        recordings_B.append({
            "recording_id": "B_" + path.stem.replace("envb_", ""),
            "label": activity,
            "environment": "Environment B",
            "source_file": path.name,
            "raw": pd.DataFrame(amplitudes).reset_index(drop=True),
        })


recordings = recordings_A + recordings_B

recording_summary = pd.DataFrame([
    {
        "recording_id": r["recording_id"],
        "label": r["label"],
        "environment": r["environment"],
        "frames": len(r["raw"]),
        "source_file": r["source_file"],
    }
    for r in recordings
]).sort_values(
    ["environment", "label", "recording_id"]
).reset_index(drop=True)

display(recording_summary)

print("Environment A:", len(recordings_A))
print("Environment B:", len(recordings_B))
print("TOTAL:", len(recordings))

display(
    recording_summary.groupby(
        ["environment", "label"]
    )["recording_id"]
    .count()
    .unstack(fill_value=0)
)

assert len(recordings_A) == 13
assert len(recordings_B) == 12
assert len(recordings) == 25

## 6. HSG preprocessing cache

Filtering does not depend on the temporal-window size, so each full/train/validation raw partition is filtered only once.

Segmentation into W10, W20, or W40 is performed later.

In [ ]:
def preprocess_partition(df):
    processed = pd.DataFrame(
        index=df.index
    )

    for col in df.columns:
        values = df[col].astype(float)

        h = hampel(
            values,
            window_size=HAMPEL_WINDOW,
        )

        smoothed = savgol_filter(
            np.asarray(
                h.filtered_data,
                dtype=float,
            ),
            window_length=SAVGOL_WINDOW,
            polyorder=SAVGOL_POLYORDER,
        )

        processed[col] = smoothed

    processed.columns = range(
        processed.shape[1]
    )

    return (
        processed
        .drop(
            columns=COLUMNS_TO_DROP,
            axis=1,
        )
        .reset_index(drop=True)
    )


cache_signature = {
    "recordings": sorted(
        (
            r["recording_id"],
            len(r["raw"]),
        )
        for r in recordings
    ),
    "train_fraction": DEV_TRAIN_FRACTION,
    "columns_to_drop": COLUMNS_TO_DROP,
    "hampel_window": HAMPEL_WINDOW,
    "savgol_window": SAVGOL_WINDOW,
    "savgol_polyorder": SAVGOL_POLYORDER,
    "version": 1,
}

processed_cache = None

if PREPROCESS_CACHE.exists():
    saved = joblib.load(
        PREPROCESS_CACHE
    )

    if saved.get("signature") == cache_signature:
        processed_cache = saved["processed_cache"]
        print("Loaded preprocessing cache:", PREPROCESS_CACHE)


if processed_cache is None:
    processed_cache = {}

    min_filter_frames = max(
        HAMPEL_WINDOW,
        SAVGOL_WINDOW,
    )

    t0 = time.time()

    for index, recording in enumerate(
        recordings,
        start=1,
    ):
        raw = recording["raw"].reset_index(drop=True)

        cut = int(
            len(raw)
            * DEV_TRAIN_FRACTION
        )

        train_raw = raw.iloc[
            :cut
        ].reset_index(drop=True)

        val_raw = raw.iloc[
            cut:
        ].reset_index(drop=True)

        if (
            len(train_raw) < min_filter_frames
            or len(val_raw) < min_filter_frames
        ):
            raise ValueError(
                f"{recording['recording_id']}: partition too short for filtering "
                f"(train={len(train_raw)}, val={len(val_raw)})"
            )

        processed_cache[
            recording["recording_id"]
        ] = {
            "label": recording["label"],
            "environment": recording["environment"],
            "full": preprocess_partition(raw),
            "train": preprocess_partition(train_raw),
            "val": preprocess_partition(val_raw),
        }

        print(
            f"[{index:02d}/25] "
            f"{recording['recording_id']} | "
            f"raw={len(raw)} | "
            f"train={len(train_raw)} | "
            f"val={len(val_raw)}"
        )

    joblib.dump(
        {
            "signature": cache_signature,
            "processed_cache": processed_cache,
        },
        PREPROCESS_CACHE,
        compress=3,
    )

    print(
        f"\nPreprocessing finished in {(time.time() - t0) / 60:.2f} min"
    )

assert len(processed_cache) == 25

## 7. Fixed 5-fold recording-disjoint plan

In [ ]:
recording_table = recording_summary[
    [
        "recording_id",
        "label",
        "environment",
    ]
].copy().reset_index(drop=True)

outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

fold_plan = []

for fold_index, (
    development_idx,
    test_idx,
) in enumerate(
    outer_cv.split(
        recording_table["recording_id"],
        recording_table["label"],
    ),
    start=1,
):
    development_ids = (
        recording_table
        .iloc[development_idx]["recording_id"]
        .tolist()
    )

    test_ids = (
        recording_table
        .iloc[test_idx]["recording_id"]
        .tolist()
    )

    fold_plan.append({
        "fold": fold_index,
        "development_ids": development_ids,
        "test_ids": test_ids,
    })


all_test_ids = [
    recording_id
    for fold in fold_plan
    for recording_id in fold["test_ids"]
]

assert len(all_test_ids) == 25
assert len(set(all_test_ids)) == 25
assert set(all_test_ids) == set(
    recording_table["recording_id"]
)

plan_rows = []

for fold in fold_plan:
    test_subset = recording_table[
        recording_table[
            "recording_id"
        ].isin(
            fold["test_ids"]
        )
    ]

    plan_rows.append({
        "fold": fold["fold"],
        "development_recordings": len(
            fold["development_ids"]
        ),
        "test_recordings": len(
            fold["test_ids"]
        ),
        "test_env_A": (
            test_subset["environment"]
            == "Environment A"
        ).sum(),
        "test_env_B": (
            test_subset["environment"]
            == "Environment B"
        ).sum(),
    })

display(
    pd.DataFrame(plan_rows)
)

print(
    "Same fold plan will be used for W10, W20 and W40."
)

## 8. Label encoder and helpers

In [ ]:
ALL_CLASSES = sorted(
    recording_summary["label"].unique()
)

label_encoder = LabelEncoder()
label_encoder.fit(
    ALL_CLASSES
)

print(
    "Class order:",
    list(label_encoder.classes_)
)


def segment_df(
    df,
    window_length,
    label,
    recording_id,
    environment,
):
    rows = []

    for start in range(
        0,
        len(df) - window_length + 1,
        STRIDE,
    ):
        end = start + window_length

        rows.append({
            "X": df.iloc[start:end].values.astype(np.float32),
            "label": label,
            "recording_id": recording_id,
            "environment": environment,
        })

    return rows


def collect_rows(
    rows
):
    X = np.stack(
        [
            row["X"]
            for row in rows
        ]
    ).astype(np.float32)

    y = label_encoder.transform(
        [
            row["label"]
            for row in rows
        ]
    ).astype(np.int64)

    return X, y


def apply_scaler(
    scaler,
    X,
    window_length,
):
    z = scaler.transform(
        X.reshape(
            len(X),
            -1,
        )
    )

    return z.reshape(
        len(X),
        window_length,
        N_FEATURES,
    ).astype(np.float32)


def build_model(
    window_length
):
    model = keras.Sequential([
        keras.Input(
            shape=(
                window_length,
                N_FEATURES,
            )
        ),
        layers.Bidirectional(
            layers.LSTM(64)
        ),
        layers.Dropout(0.5),
        layers.Dense(
            len(label_encoder.classes_),
            activation="softmax",
        ),
    ])

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

## 9. Run the 3 × 5 window-ablation experiment

Total training runs:

```text
3 configurations × 5 folds = 15 models
```

In [ ]:
summary_rows = []
fold_rows = []
prediction_tables = []

experiment_start = time.time()
completed_runs = 0
total_runs = len(CONFIGS) * len(fold_plan)

for config in CONFIGS:
    config_id = config["config_id"]
    window_length = config["window"]

    print(
        "\n"
        + "#" * 90
    )
    print(
        f"CONFIG {config_id}: "
        f"{window_length} frames | Hampel + SG"
    )
    print(
        "#" * 90
    )

    config_predictions = []

    for fold in fold_plan:
        fold_index = fold["fold"]
        development_ids = fold["development_ids"]
        test_ids = fold["test_ids"]

        train_rows = []
        val_rows = []
        test_rows = []

        for recording_id in development_ids:
            item = processed_cache[
                recording_id
            ]

            train_rows.extend(
                segment_df(
                    item["train"],
                    window_length,
                    item["label"],
                    recording_id,
                    item["environment"],
                )
            )

            val_rows.extend(
                segment_df(
                    item["val"],
                    window_length,
                    item["label"],
                    recording_id,
                    item["environment"],
                )
            )

        for recording_id in test_ids:
            item = processed_cache[
                recording_id
            ]

            test_rows.extend(
                segment_df(
                    item["full"],
                    window_length,
                    item["label"],
                    recording_id,
                    item["environment"],
                )
            )

        if len(train_rows) == 0:
            raise ValueError(
                f"{config_id} fold {fold_index}: no training windows"
            )

        if len(val_rows) == 0:
            raise ValueError(
                f"{config_id} fold {fold_index}: no validation windows"
            )

        if len(test_rows) == 0:
            raise ValueError(
                f"{config_id} fold {fold_index}: no test windows"
            )

        train_ids = {
            row["recording_id"]
            for row in train_rows
        }

        val_ids = {
            row["recording_id"]
            for row in val_rows
        }

        test_recording_ids = {
            row["recording_id"]
            for row in test_rows
        }

        assert (
            train_ids
            | val_ids
        ).isdisjoint(
            test_recording_ids
        )

        X_train_raw, y_train = collect_rows(
            train_rows
        )

        X_val_raw, y_val = collect_rows(
            val_rows
        )

        X_test_raw, y_test = collect_rows(
            test_rows
        )

        scaler = StandardScaler()

        scaler.fit(
            X_train_raw.reshape(
                len(X_train_raw),
                -1,
            )
        )

        X_train = apply_scaler(
            scaler,
            X_train_raw,
            window_length,
        )

        X_val = apply_scaler(
            scaler,
            X_val_raw,
            window_length,
        )

        X_test = apply_scaler(
            scaler,
            X_test_raw,
            window_length,
        )

        keras.backend.clear_session()

        random.seed(SEED)
        np.random.seed(SEED)
        tf.keras.utils.set_random_seed(SEED)

        model = build_model(
            window_length
        )

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
        )

        run_start = time.time()

        history = model.fit(
            X_train,
            y_train,
            validation_data=(
                X_val,
                y_val,
            ),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[
                early_stopping
            ],
            shuffle=True,
            verbose=1,
        )

        probabilities = model.predict(
            X_test,
            verbose=0,
        )

        y_pred = np.argmax(
            probabilities,
            axis=1,
        )

        accuracy = accuracy_score(
            y_test,
            y_pred,
        )

        macro_f1 = f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        )

        weighted_f1 = f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0,
        )

        best_epoch = int(
            np.argmin(
                history.history[
                    "val_loss"
                ]
            )
            + 1
        )

        fold_rows.append({
            "config_id": config_id,
            "window": window_length,
            "fold": fold_index,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "best_epoch": best_epoch,
            "train_windows": len(
                y_train
            ),
            "validation_windows": len(
                y_val
            ),
            "test_windows": len(
                y_test
            ),
            "minutes": (
                time.time()
                - run_start
            )
            / 60,
        })

        metadata = pd.DataFrame([
            {
                "recording_id": row[
                    "recording_id"
                ],
                "environment": row[
                    "environment"
                ],
                "true_label": row[
                    "label"
                ],
            }
            for row in test_rows
        ])

        metadata[
            "config_id"
        ] = config_id

        metadata[
            "window"
        ] = window_length

        metadata[
            "fold"
        ] = fold_index

        metadata[
            "predicted_label"
        ] = label_encoder.inverse_transform(
            y_pred
        )

        metadata[
            "confidence"
        ] = np.max(
            probabilities,
            axis=1,
        )

        config_predictions.append(
            metadata
        )

        completed_runs += 1

        elapsed = (
            time.time()
            - experiment_start
        ) / 60

        remaining = (
            elapsed
            / completed_runs
        ) * (
            total_runs
            - completed_runs
        )

        print(
            f"{config_id} | "
            f"fold {fold_index}/5 | "
            f"acc={accuracy:.4f} | "
            f"macroF1={macro_f1:.4f} | "
            f"best_epoch={best_epoch} | "
            f"remaining≈{remaining:.1f} min"
        )

    config_predictions = pd.concat(
        config_predictions,
        ignore_index=True,
    )

    prediction_tables.append(
        config_predictions
    )

    y_true_all = config_predictions[
        "true_label"
    ].to_numpy()

    y_pred_all = config_predictions[
        "predicted_label"
    ].to_numpy()

    overall_accuracy = accuracy_score(
        y_true_all,
        y_pred_all,
    )

    overall_macro_f1 = f1_score(
        y_true_all,
        y_pred_all,
        average="macro",
        zero_division=0,
    )

    overall_weighted_f1 = f1_score(
        y_true_all,
        y_pred_all,
        average="weighted",
        zero_division=0,
    )

    env_metrics = {}

    for environment in [
        "Environment A",
        "Environment B",
    ]:
        subset = config_predictions[
            config_predictions[
                "environment"
            ]
            == environment
        ]

        env_true = subset[
            "true_label"
        ].to_numpy()

        env_pred = subset[
            "predicted_label"
        ].to_numpy()

        env_metrics[
            environment
        ] = {
            "accuracy": accuracy_score(
                env_true,
                env_pred,
            ),
            "macro_f1": f1_score(
                env_true,
                env_pred,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                env_true,
                env_pred,
                average="weighted",
                zero_division=0,
            ),
        }

    config_fold_df = pd.DataFrame(
        [
            row
            for row in fold_rows
            if row[
                "config_id"
            ]
            == config_id
        ]
    )

    summary_rows.append({
        "config_id": config_id,
        "configuration": (
            f"{window_length} | Hampel + SG"
        ),
        "window": window_length,
        "accuracy": overall_accuracy,
        "macro_f1": overall_macro_f1,
        "weighted_f1": overall_weighted_f1,
        "fold_accuracy_mean": config_fold_df[
            "accuracy"
        ].mean(),
        "fold_accuracy_std": config_fold_df[
            "accuracy"
        ].std(
            ddof=1
        ),
        "fold_macro_f1_mean": config_fold_df[
            "macro_f1"
        ].mean(),
        "fold_macro_f1_std": config_fold_df[
            "macro_f1"
        ].std(
            ddof=1
        ),
        "accuracy_A": env_metrics[
            "Environment A"
        ][
            "accuracy"
        ],
        "macro_f1_A": env_metrics[
            "Environment A"
        ][
            "macro_f1"
        ],
        "weighted_f1_A": env_metrics[
            "Environment A"
        ][
            "weighted_f1"
        ],
        "accuracy_B": env_metrics[
            "Environment B"
        ][
            "accuracy"
        ],
        "macro_f1_B": env_metrics[
            "Environment B"
        ][
            "macro_f1"
        ],
        "weighted_f1_B": env_metrics[
            "Environment B"
        ][
            "weighted_f1"
        ],
        "balanced_environment_macro_f1": (
            env_metrics[
                "Environment A"
            ][
                "macro_f1"
            ]
            + env_metrics[
                "Environment B"
            ][
                "macro_f1"
            ]
        )
        / 2,
    })


summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    "window"
).reset_index(
    drop=True
)

fold_metrics_df = pd.DataFrame(
    fold_rows
)

all_predictions_df = pd.concat(
    prediction_tables,
    ignore_index=True,
)

display(
    summary_df
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

fold_metrics_df.to_csv(
    FOLD_CSV,
    index=False,
)

all_predictions_df.to_csv(
    PREDICTIONS_CSV,
    index=False,
)

## 10. Window-ablation plots

In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 5)
)

ax.plot(
    summary_df[
        "window"
    ],
    summary_df[
        "accuracy"
    ],
    marker="o",
    label="Accuracy",
)

ax.plot(
    summary_df[
        "window"
    ],
    summary_df[
        "macro_f1"
    ],
    marker="o",
    label="Macro F1",
)

ax.set_xlabel(
    "Window length [frames]"
)

ax.set_ylabel(
    "Score"
)

ax.set_ylim(
    0,
    1,
)

ax.set_title(
    "A13 + B12 — recording-disjoint window ablation"
)

ax.grid(
    True,
    alpha=0.3,
)

ax.legend()

fig.tight_layout()

fig.savefig(
    OVERALL_PDF,
    bbox_inches="tight",
)

plt.show()


fig, ax = plt.subplots(
    figsize=(7, 5)
)

ax.plot(
    summary_df[
        "window"
    ],
    summary_df[
        "macro_f1_A"
    ],
    marker="o",
    label="Environment A Macro F1",
)

ax.plot(
    summary_df[
        "window"
    ],
    summary_df[
        "macro_f1_B"
    ],
    marker="o",
    label="Environment B Macro F1",
)

ax.set_xlabel(
    "Window length [frames]"
)

ax.set_ylabel(
    "Macro F1"
)

ax.set_ylim(
    0,
    1,
)

ax.set_title(
    "Window ablation by environment"
)

ax.grid(
    True,
    alpha=0.3,
)

ax.legend()

fig.tight_layout()

fig.savefig(
    ENV_PDF,
    bbox_inches="tight",
)

plt.show()

## 11. Compact article-ready table

Use this table to decide whether W40 still provides the strongest result on the final A13+B12 dataset.

In [ ]:
article_table = summary_df[
    [
        "config_id",
        "window",
        "accuracy",
        "macro_f1",
        "accuracy_A",
        "macro_f1_A",
        "accuracy_B",
        "macro_f1_B",
    ]
].copy()

display(
    article_table
)

best_by_macro = summary_df.sort_values(
    "macro_f1",
    ascending=False,
).iloc[0]

print(
    "Highest overall Macro F1:",
    best_by_macro[
        "config_id"
    ],
    f"({best_by_macro['macro_f1']:.4f})",
)

## 12. Package results

In [ ]:
result_files = [
    SUMMARY_CSV,
    FOLD_CSV,
    PREDICTIONS_CSV,
    OVERALL_PDF,
    ENV_PDF,
]

with zipfile.ZipFile(
    RESULT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file_path in result_files:
        if file_path.exists():
            zf.write(
                file_path,
                arcname=file_path.name,
            )

print(
    "Created:",
    RESULT_ZIP,
)

try:
    from google.colab import files

    files.download(
        str(
            RESULT_ZIP
        )
    )

except ImportError:
    pass